In [1]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate scikit-learn
from huggingface_hub import login
login()

In [5]:
import torch, json, os, random, gc, numpy as np
from google.colab import drive
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/finsent"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")

# Manually define the class labels corresponding to the numeric representations
label_words = ["Bearish", "Bullish", "Neutral"]

try:
    split = ds["train"].train_test_split(test_size=0.1, seed=SEED, stratify_by_column="label")
except ValueError:
    split = ds["train"].train_test_split(test_size=0.1, seed=SEED)
train_ds, dev_ds = split["train"], split["test"]
print(f"Loaded train size: {len(train_ds)}, dev size: {len(dev_ds)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded train size: 8588, dev size: 955


In [6]:
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token = tok.eos_token
tok.padding_side = "right"
PROMPT_DATE = "01 Jan 2026"

def build_prompt(text):
    msgs = [{"role": "system", "content": "You are a financial sentiment classifier."},
            {"role": "user", "content": f"Classify the sentiment of this finance tweet as exactly one of: {', '.join(label_words)}.\n\nTweet: {text}\n\nAnswer with one word."}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, date_string=PROMPT_DATE)
print(build_prompt("test")[:200])   # must show Today Date: 01 Jan 2026

def encode(ex):
    p = tok(build_prompt(ex["text"]), add_special_tokens=False)["input_ids"]
    a = tok(label_words[ex["label"]] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return {"input_ids": p + a, "attention_mask": [1]*(len(p)+len(a)), "labels": [-100]*len(p) + a}

train_tok = train_ds.map(encode, remove_columns=train_ds.column_names)
dev_tok = dev_ds.map(encode, remove_columns=dev_ds.column_names)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map="auto", dtype=torch.float16)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

R = 8   # <-- the ONE change (main model uses 16); alpha=2*r keeps alpha/r = 2
lora = LoraConfig(r=R, lora_alpha=2*R, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 01 Jan 2026

You are a financial sentiment classifier.<|eot_id|><|start_header_id|>user<|


Map:   0%|          | 0/8588 [00:00<?, ? examples/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 12,156,928 || all params: 3,224,906,752 || trainable%: 0.3770


In [7]:
SMOKE = False   # set True first for a 30-step test, then False
RUN = f"r{R}_lr2e-4" + ("_smoke" if SMOKE else "")
CK = f"{OUT}/checkpoints_{RUN}"

args = TrainingArguments(
    output_dir=CK,
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    learning_rate=2e-4, num_train_epochs=2, max_steps=30 if SMOKE else -1,
    lr_scheduler_type="cosine", warmup_steps=30,
    fp16=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10, eval_strategy="steps", eval_steps=200,
    save_strategy="steps", save_steps=200, save_total_limit=2,
    report_to="none", seed=SEED,
)
collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100, return_tensors="pt")
trainer = Trainer(model=model, args=args, train_dataset=train_tok,
                  eval_dataset=dev_tok.select(range(300)), data_collator=collator)

resume = True if os.path.isdir(CK) and any(d.startswith("checkpoint-") for d in os.listdir(CK)) else None
trainer.train(resume_from_checkpoint=resume)

trainer.model.save_pretrained(f"{OUT}/adapter_{RUN}")
tok.save_pretrained(f"{OUT}/adapter_{RUN}")
json.dump(trainer.state.log_history, open(f"{OUT}/log_{RUN}.json", "w"))
print("saved", RUN)

Step,Training Loss,Validation Loss
200,0.107597,0.154829
400,0.130254,0.145537
600,0.062537,0.104090
800,0.059135,0.096329
1000,0.037528,0.090258
1074,0.046494,0.089750


saved r8_lr2e-4


In [8]:
# Evaluate the variant on the 955 DEV split
tok.padding_side = "left"
label_token_ids = [tok.encode(w, add_special_tokens=False)[0] for w in label_words]
dev_texts, dev_y = dev_ds["text"], dev_ds["label"]

@torch.no_grad()
def predict(m, texts, bs=16):
    m.eval(); preds = []
    for i in tqdm(range(0, len(texts), bs)):
        enc = tok([build_prompt(t) for t in texts[i:i+bs]], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(m.device)
        preds += m(**enc).logits[:, -1, :][:, label_token_ids].argmax(-1).cpu().tolist()
    return preds

def score(name, preds):
    r = {"name": name, "n": len(dev_y), "accuracy": accuracy_score(dev_y, preds),
         "macro_f1": f1_score(dev_y, preds, average="macro"),
         "confusion_matrix": confusion_matrix(dev_y, preds).tolist()}
    print(name, "| acc", round(r["accuracy"],4), "| macro-F1", round(r["macro_f1"],4)); print(r["confusion_matrix"])
    return r

variant = score(f"{RUN}_dev", predict(trainer.model, dev_texts))

# free GPU, then load base + MAIN adapter (r16) for a like-for-like dev comparison
del trainer, model; gc.collect(); torch.cuda.empty_cache()
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                            device_map={"": 0}, dtype=torch.float16)
main = PeftModel.from_pretrained(base, f"{OUT}/adapter_r16_lr2e-4")
main_res = score("r16_lr2e-4_dev", predict(main, dev_texts))

json.dump({"variant": variant, "main": main_res}, open(f"{OUT}/ablation_dev_r8_vs_r16.json", "w"))
print("Report as an ablation; main model stays r16_lr2e-4.")

  0%|          | 0/60 [00:00<?, ?it/s]

r8_lr2e-4_dev | acc 0.9152 | macro-F1 0.8876
[[109, 5, 22], [1, 175, 21], [16, 16, 590]]


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

r16_lr2e-4_dev | acc 0.9225 | macro-F1 0.8992
[[116, 4, 16], [2, 173, 22], [15, 15, 592]]
Report as an ablation; main model stays r16_lr2e-4.
